## Installing initial dependencies

In [ ]:
!pip install gdown. 

In [ ]:
import gdown

file_id = "11hRKwkFA46mD5W2OCoM-FvXTsn-73Zw6"  # no trailing slash
output_file = "individual_files_testing_segment.tfrecord"

gdown.download(f"https://drive.google.com/uc?id={file_id}", output_file, quiet=False)

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install tensorflow==2.10.0
!{sys.executable} -m pip install waymo-open-dataset-tf-2-12-0

In [ ]:
import tensorflow as tf
print("TF version:", tf.__version__)

from waymo_open_dataset import dataset_pb2 as open_dataset
from waymo_open_dataset.utils import frame_utils

print("Waymo Open Dataset working.")

# Installing cupy for a gpu based numpy hadling -- Comment out this line if we are not using a cuda graphic card
!pip install cupy --pre

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

TF version: 2.12.0
Waymo Open Dataset working.


## Creating audit report for all the t_records present in the folder

In [ ]:
import os
import time
import cupy as cp
import numpy as np
import csv
import tensorflow.compat.v1 as tf
tf.enable_eager_execution()
from PIL import Image

tf.enable_eager_execution()

from waymo_open_dataset.utils import frame_utils
from waymo_open_dataset import dataset_pb2 as open_dataset

# CSV CONFIG
DATASET_DIR = "/workspace/tfrecords"  
CSV_OUTPUT_DIR = "/workspace/audit"

# Thresholds
DENSITY_MIN = 0.6
DENSITY_MAX = 2
QUALIFY_THRESHOLD = 0.9

# Creating the audit folder if it was not already existing
os.makedirs(CSV_OUTPUT_DIR, exist_ok=True)

def compute_frame_density(frame) -> float | None:
    """Compute FRONT-camera LiDAR density and depth map using GPU (CuPy)."""
    # Decode FRONT camera
    front_image = None
    for img in frame.images:
        if img.name == open_dataset.CameraName.FRONT:
            front_image = tf.image.decode_jpeg(img.image).numpy()
            break
    if front_image is None:
        return None

    H, W = front_image.shape[:2]
    total_pixels = H * W

    # Parse range images
    range_images, camera_projections, _, range_image_top_pose = \
        frame_utils.parse_range_image_and_camera_projection(frame)

    points_ri1, cp_ri1 = frame_utils.convert_range_image_to_point_cloud(
        frame, range_images, camera_projections, range_image_top_pose, ri_index=0)
    points_ri2, cp_ri2 = frame_utils.convert_range_image_to_point_cloud(
        frame, range_images, camera_projections, range_image_top_pose, ri_index=1)

    points_all = np.concatenate(points_ri1 + points_ri2, axis=0)
    cp_all     = np.concatenate(cp_ri1 + cp_ri2, axis=0)

    # Extrinsic for FRONT camera
    extrinsic = None
    for calib in frame.context.camera_calibrations:
        if calib.name == open_dataset.CameraName.FRONT:
            extrinsic = np.array(calib.extrinsic.transform).reshape(4, 4)
            break
    if extrinsic is None:
        return None

    # Move points to GPU
    points_gpu = cp.asarray(points_all)
    cp_gpu     = cp.asarray(cp_all)

    front_mask = cp_gpu[:, 0] == open_dataset.CameraName.FRONT
    front_points = points_gpu[front_mask]
    u = cp_gpu[front_mask][:, 1].astype(cp.int32)
    v = cp_gpu[front_mask][:, 2].astype(cp.int32)

    # Transform to camera space
    ones = cp.ones((front_points.shape[0], 1), dtype=cp.float32)
    front_h = cp.concatenate([front_points, ones], axis=1)
    vehicle_to_cam = cp.asarray(np.linalg.inv(extrinsic), dtype=cp.float32)
    points_cam = front_h @ vehicle_to_cam.T
    z = points_cam[:, 2]

    # Valid pixels
    valid = (u >= 0) & (u < W) & (v >= 0) & (v < H) & (z > 0)
    u, v, z = u[valid], v[valid], z[valid]

    # Depth map on GPU
    depth_map = cp.full((H, W), cp.inf, dtype=cp.float32)
    cp.scatter_min(depth_map, (v, u), z)
    depth_map = cp.where(depth_map == cp.inf, cp.nan, depth_map)

    # Compute density
    valid_pixel_count = int(cp.sum(~cp.isnan(depth_map)))
    density_pct = (valid_pixel_count / total_pixels) * 100

    return density_pct


# Function to create the dict array 
def audit_tfrecord(tfrecord_path: str) -> dict:
    segment_name = os.path.splitext(os.path.basename(tfrecord_path))[0]
    dataset = tf.data.TFRecordDataset(tfrecord_path, compression_type='')

    densities = []
    frame_idx = 0
    t0 = time.time()

    for raw_data in dataset:
        frame = open_dataset.Frame()
        frame.ParseFromString(bytearray(raw_data.numpy()))
        density = compute_frame_density(frame)

        if density is not None:
            densities.append(density)
            if True:
                band = "✓" if DENSITY_MIN <= density <= DENSITY_MAX else "✗"
                print(f"    frame {frame_idx:04d} | density={density:.3f}% {band}")
        frame_idx += 1

    elapsed = time.time() - t0

    if not densities:
        return {
            "segment": segment_name, "total_frames": frame_idx,
            "sampled_frames": 0, "good_frames": 0, "good_pct": 0.0,
            "mean_density": 0.0, "min_density": 0.0, "max_density": 0.0,
            "median_density": 0.0, "label": "NOT SUITABLE",
            "reason": "no valid frames extracted", "elapsed_s": round(elapsed,1)
        }

    densities_arr = np.array(densities)
    good_mask = (densities_arr >= DENSITY_MIN) & (densities_arr <= DENSITY_MAX)
    good_frames = int(np.sum(good_mask))
    good_pct = good_frames / len(densities)

    if good_pct >= QUALIFY_THRESHOLD:
        label = "SUITABLE"
        reason = f"{good_pct*100:.1f}% frames within [{DENSITY_MIN}%, {DENSITY_MAX}%]"
    else:
        dead = int(np.sum(densities_arr < DENSITY_MIN))
        too_high = int(np.sum(densities_arr > DENSITY_MAX))
        label = "NOT SUITABLE"
        reason = f"{good_pct*100:.1f}% qualify (dead<{DENSITY_MIN}%: {dead}, above {DENSITY_MAX}%: {too_high})"

    return {
        "segment": segment_name, "total_frames": frame_idx,
        "sampled_frames": len(densities), "good_frames": good_frames,
        "good_pct": round(good_pct*100,2),
        "mean_density": round(float(np.mean(densities_arr)),4),
        "min_density": round(float(np.min(densities_arr)),4),
        "max_density": round(float(np.max(densities_arr)),4),
        "median_density": round(float(np.median(densities_arr)),4),
        "label": label, "reason": reason, "elapsed_s": round(elapsed,1)
    }

# RUN AUDIT
results = []
tfrecord_files = sorted(
    os.path.join(DATASET_DIR, f) for f in os.listdir(DATASET_DIR) if f.endswith(".tfrecord")
)

for i, tf_path in enumerate(tfrecord_files):
    print(f"[{i+1}/{len(tfrecord_files)}] {os.path.basename(tf_path)}")
    row = audit_tfrecord(tf_path)
    results.append(row)
    print(f"  → {row['label']} | {row['reason']} | mean={row['mean_density']:.3f}% ({row['elapsed_s']}s)\n")

# Write CSV
csv_path = os.path.join(CSV_OUTPUT_DIR, "waymo_audit_results.csv")
fieldnames = ["segment","label","good_pct","mean_density","min_density","max_density",
              "median_density","good_frames","sampled_frames","total_frames","reason","elapsed_s"]
with open(csv_path,"w",newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

# Write summary
summary_path = os.path.join(CSV_OUTPUT_DIR, "waymo_audit_summary.txt")
with open(summary_path,"w") as f:
    f.write("WAYMO DENSITY AUDIT SUMMARY\n")
    f.write(f"Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Density band: [{DENSITY_MIN}%, {DENSITY_MAX}%] | threshold={QUALIFY_THRESHOLD*100:.0f}%\n")
    f.write("─"*60+"\n\n")
    suitable = [r for r in results if r["label"]=="SUITABLE"]
    not_suitable = [r for r in results if r["label"]!="SUITABLE"]
    f.write(f"SUITABLE ({len(suitable)}/{len(results)})\n")
    for r in suitable:
        f.write(f"  {r['segment']}\n")
        f.write(f"    good={r['good_pct']:.1f}% mean={r['mean_density']:.3f}% "
                f"[{r['min_density']:.3f}%, {r['max_density']:.3f}%]\n")
    f.write(f"\nNOT SUITABLE ({len(not_suitable)}/{len(results)})\n")
    for r in not_suitable:
        f.write(f"  {r['segment']}\n    {r['reason']}\n")

print(f"DONE. CSV: {csv_path} | Summary: {summary_path}")
